# 29 · Every published model on one set

The numbers in `summary.md` were measured on four different sets: notebook 0's patches (notebooks 1–16, a set no
longer on disk), notebook 17's (18–20), notebook 21's (22–27), and notebook 2's older synthetic frames. A score on one
cannot be set beside a score on another. This notebook scores **every model in `dist/upscale/` on one set**: notebook
28's validation split, 24,216 patches of video, drawn screens and photos, through both of the host's encoders at slice
QPs from 0 to 51.

- **The graphs, not the training code.** Notebooks 1–16 left no checkpoints, only their exported graphs, so every
  model is scored as the client runs it: its `.onnx`, through onnxruntime. A tile graph takes a fixed picture, so
  several patches are packed into one tile, each with its own halo of real pixels, and each patch's 128 px core is cut
  from the output. Section 2 checks that this is exact: a core must not change with what sits beside it in the tile.
- **One model, one graph.** A model exported several times (float16, whole-frame, other tile sizes) is scored once,
  as its float32 tile graph or the client's graph.
- **Told graphs are told what the client reads**: each patch's slice QP / 51, loop filter and frame type. Notebook 8's
  graphs were told a bitrate when they were exported and are scored as they are.
- **Scores as every notebook takes them**: `metrics.per_image` on the core, SSIM-Y and luma PSNR, a mean over the
  patches; a gain is against `models.bicubic`, the skip the models are built on. The gains carry a 95% range from a
  bootstrap over the set's sources (an image, or a video sequence), since the patches of one source move together.
- **A check first**: on notebook 21's validation split the same code must give the numbers notebooks 23 to 25 recorded
  there (bicubic 0.9045, the told 24×2 0.9289, q24 0.9212).

**Limits.** No model here was trained on notebook 28's training split: notebooks 2–16 trained on notebook 0's set
(x264 only, at 0.2–1 bpp), notebooks 23–24 on notebook 21's (YouTube video only). This scores the stream a client
meets, not each model on its own validation. Notebooks 0 and 21 picked best epochs on validation sources that are also
in this set (YouTube-8M's test videos, the first set's test screens, DIV2K valid). Patches are scored on their 64 px
core; tile seams and whole frames are the benchmark page's and notebook 15's.

Needs: notebook 28 (`qp_diverse_pairs/`), notebook 21 (`qp_slice_pairs/`). Writes per-patch scores to `scoreboard/`;
trains and exports nothing.

In [1]:
"""Settings."""

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training
from helpers.training import DEVICE

DIST = Path(sources.__file__).resolve().parents[3] / "dist" / "upscale"
SETS = {"28": sources.ROOT / "upscale" / "qp_diverse_pairs",   # the one set: notebook 28's validation split
        "21": sources.ROOT / "upscale" / "qp_slice_pairs"}     # notebook 21's, to check against the numbers recorded on it
CACHE = sources.ROOT / "upscale" / "scoreboard"   # per-patch scores, a file a graph and set, named by the graph's hash

# (notebook, graph in DIST, name): one graph a model
GRAPHS = [
    (1, "upscale_static_nearest_tile128.onnx", "nearest (Resize)"),
    (1, "upscale_static_bilinear_tile320x180.onnx", "bilinear (Resize)"),
    (1, "upscale_static_bicubic_tile320x180.onnx", "bicubic (Resize)"),
    (2, "upscale_nn_tile128.onnx", "plain CNN 16-24-24-32"),
    (3, "upscale_web_tile128.onnx", "web 32-16"),
    (3, "upscale_web_int8_tile128.onnx", "web 32-16, int8"),
    (4, "upscale_strategy_32-16_leaky_tile320x180.onnx", "strategy 32-16 leaky"),
    (4, "upscale_strategy_luma_24_2_blocks_tile320x180.onnx", "strategy luma 24 + 2 blocks"),
    (4, "upscale_strategy_luma_32-16_tile320x180.onnx", "strategy luma 32-16"),
    (6, "upscale_reparam_32-16_tile320x180.onnx", "reparam 32-16"),
    (7, "upscale_luma_24x2_tile320x180.onnx", "luma 24x2"),
    (8, "upscale_screens_24x2_bpp0.2_tile320x180.onnx", "screens 24x2, told 0.2 bpp"),
    (8, "upscale_screens_24x2_bpp0.46_tile320x180.onnx", "screens 24x2, told 0.46 bpp"),
    (8, "upscale_screens_24x2_bpp1_tile320x180.onnx", "screens 24x2, told 1 bpp"),
    (9, "upscale_dct_fixed_tile320x180.onnx", "DCT fixed"),
    (9, "upscale_dct_h32_tile320x180.onnx", "DCT h32"),
    (10, "upscale_lut_s_17_tile320x180.onnx", "LUT S 17"),
    (10, "upscale_lut_sdy_x2_17_tile320x180.onnx", "LUT SDY x2 17"),
    (11, "upscale_fastlut_s_tile320x180.onnx", "fast LUT S"),
    (11, "upscale_fastlut_sdyx2_tile320x180.onnx", "fast LUT SDY x2"),
    (11, "upscale_fastlut_3x3pattern_tile320x180.onnx", "fast LUT 3x3 pattern"),
    (12, "upscale_dirnet_fixed_tile320x180.onnx", "DirNet fixed"),
    (12, "upscale_dirnet_adaptive_tile320x180.onnx", "DirNet adaptive"),
    (12, "upscale_dirnet_full_tile320x180.onnx", "DirNet full"),
    (16, "upscale_luma_32-16_leaky_tile320x180.onnx", "luma 32-16 leaky, distilled"),
    (23, "client/upscale.onnx", "told 24x2 (the client's)"),
    (24, "client/upscale_fast.onnx", "told q24 fast"),
]
RECORDED_21 = {"bicubic": (0.9045, 33.55), "client/upscale.onnx": (0.9289, 35.07),   # notebooks 25, 24
               "client/upscale_fast.onnx": (0.9212, 34.47)}
QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46, 52)
CHUNK, THREADS, BOOTSTRAP = 2048, 16, 2000

CACHE.mkdir(parents=True, exist_ok=True)
print("graphs", DIST, " device", DEVICE, " onnxruntime", ort.__version__)

graphs /home/henrik/projects/desktop-streamer-ai/dist/upscale  device cuda  onnxruntime 1.29.0


In [2]:
def twin_of(name):
    """Why a graph in DIST is not scored: the model it holds is scored as another file."""
    if name.endswith("_frame.onnx"):
        return "whole-frame twin of a tile graph"
    if "fp16" in name or name == "upscale_webbig_tile256.onnx":   # webbig: web 32-16's float16 weights at a 256 px step
        return "float16 twin"
    if name in ("upscale_static_bicubic_tile128.onnx", "upscale_static_bilinear_tile128.onnx"):
        return "the same Resize at another tile size"
    if name == "upscale_qp_slice_24x2_tile320x180.onnx":
        return "tile twin of client/upscale.onnx"
    if name.startswith("upscale_fast_q24_frozen_skip"):
        return "tile twin of client/upscale_fast.onnx"
    return None


scored = {name for _, name, _ in GRAPHS}
others = sorted(path.relative_to(DIST).as_posix() for path in DIST.rglob("*.onnx") if path.relative_to(DIST).as_posix() not in scored)
unexplained = [name for name in others if twin_of(name) is None]
assert not unexplained, f"graphs neither scored nor a twin: {unexplained}"
assert all((DIST / name).exists() for name in scored)
print(f"{len(scored)} graphs scored; {len(others)} not, each a twin of one that is:")
print(pd.Series({name: twin_of(name) for name in others}).value_counts().to_string())

27 graphs scored; 44 not, each a twin of one that is:
float16 twin                             21
whole-frame twin of a tile graph         19
the same Resize at another tile size      2
tile twin of client/upscale_fast.onnx     1
tile twin of client/upscale.onnx          1


## 1. The sets

Both validation splits, with what a told graph needs (slice QP, loop filter, I or P), the encoder, the kind (notebook
21's are all video) and the source of each patch, for the bootstrap.

In [3]:
data = {}
for key, folder in SETS.items():
    manifest = training.load_manifest(folder)
    files = manifest["splits"]["val"]
    lr, hr = patches.load(folder, "val", manifest)
    labels = {name: np.load(folder / files[name]) for name in ("qp", "deblocked", "intra", "encoder")}
    kinds = np.array(manifest.get("kinds", ["video"]))
    kind = kinds[np.load(folder / files["kind"])] if "kind" in files else np.full(len(lr), "video")
    source = np.empty(len(lr), np.int32)
    keys = {}
    for entry in files["sources"]:
        first, end = entry["patches"]
        source[first:end] = keys.setdefault(entry["key"], len(keys))
    data[key] = {"lr": lr, "hr": hr, **labels, "kind": kind, "encoder_name": np.array(manifest["encoders"])[labels["encoder"]],
                 "source": source, "halo": manifest["halo"], "scale": manifest["scale"]}
    print(f"notebook {key}'s validation: {len(lr)} patches from {len(keys)} sources, LR {lr.shape[1:]} HR {hr.shape[1:]}; "
          + ", ".join(f"{k} {(kind == k).sum()}" for k in kinds))

notebook 28's validation: 24216 patches from 426 sources, LR (72, 72, 3) HR (128, 128, 3); video 9424, screen 11142, photo 3650
notebook 21's validation: 9628 patches from 88 sources, LR (72, 72, 3) HR (128, 128, 3); video 9628


## 2. Running a graph on patches

A graph's halo is what it needs around a core; a stored patch has 4 px, cut to the graph's. A graph with a free size
(the client's) takes patches as a batch. A tile graph takes a canvas of its fixed size: the patches are laid side by
side in it, the rest left black, and each core is cut from the output at its place. A told graph is run once a
condition, on the patches that share it.

The check: a sample of patches run as scored, and again laid in the reverse order with the empty cells filled with
noise. A graph whose core sees no further than its halo gives the same cores both ways.

In [4]:
def graph_info(path):
    """What `run` needs to know of a graph: its halo, its input's size (None where any) and type, and its inputs."""
    model = onnx.load(str(path), load_external_data=False)
    meta = {prop.key: prop.value for prop in model.metadata_props}
    stored = {init.name for init in model.graph.initializer}
    inputs = [value for value in model.graph.input if value.name not in stored]
    image = inputs[0].type.tensor_type
    return {"halo": int(meta["halo"]), "dims": [dim.dim_value or None for dim in image.shape.dim],
            "dtype": np.float16 if image.elem_type == onnx.TensorProto.FLOAT16 else np.float32,
            "image": inputs[0].name, "condition": inputs[1].name if len(inputs) > 1 else None}


def open_graph(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


def run(graph, info, lr, conditions, stored_halo, scale, reverse=False, noise=None):
    """The graph's core (N, 3, 128, 128) float32 for LR patches (N, h, w, 3) uint8, under their conditions (N, 3)."""
    halo = info["halo"]
    cut = stored_halo - halo
    x = lr[:, cut:lr.shape[1] - cut, cut:lr.shape[2] - cut].astype(np.float32).transpose(0, 3, 1, 2) / 255
    cell, core = x.shape[-1], (x.shape[-1] - 2 * halo) * scale
    out = np.empty((len(x), 3, core, core), np.float32)
    told = info["condition"] is not None
    groups = [np.flatnonzero((conditions == row).all(axis=1)) for row in np.unique(conditions, axis=0)] if told else [np.arange(len(x))]
    for members in groups:
        feed = {info["condition"]: conditions[members[0]].astype(np.float32)} if told else {}
        if info["dims"][2] is None:   # any size: a batch of patches
            for first in range(0, len(members), 256):
                chosen = members[first:first + 256]
                y = graph.run(None, {**feed, info["image"]: x[chosen].astype(info["dtype"])})[0].astype(np.float32)
                out[chosen] = y[:, :, halo * scale:halo * scale + core, halo * scale:halo * scale + core]
            continue
        batch, _, height, width = info["dims"]
        rows, columns = height // cell, width // cell
        places = [(b, r, c) for b in range(batch) for r in range(rows) for c in range(columns)]
        if reverse:
            places = places[::-1]
        for first in range(0, len(members), len(places)):
            chosen = members[first:first + len(places)]
            canvas = np.zeros((batch, 3, height, width), np.float32) if noise is None else noise.random((batch, 3, height, width), np.float32)
            for i, (b, r, c) in zip(chosen, places):
                canvas[b, :, r * cell:(r + 1) * cell, c * cell:(c + 1) * cell] = x[i]
            y = graph.run(None, {**feed, info["image"]: canvas.astype(info["dtype"])})[0].astype(np.float32)
            for i, (b, r, c) in zip(chosen, places):
                top, left = (r * cell + halo) * scale, (c * cell + halo) * scale
                out[i] = y[b, :, top:top + core, left:left + core]
    return out


def conditions_of(d, members):
    """What a told graph is given for each patch: slice QP / 51, loop filter, I frame."""
    return np.stack([d["qp"][members] / 51, d["deblocked"][members], d["intra"][members]], axis=1).astype(np.float32)


check = data["28"]
sample = np.random.default_rng(0).choice(len(check["lr"]), 96, replace=False)
rows = []
for notebook, name, label in GRAPHS:
    info = graph_info(DIST / name)
    graph = open_graph(DIST / name)
    args = (graph, info, check["lr"][sample], conditions_of(check, sample), check["halo"], check["scale"])
    start = time.time()
    packed = run(*args)
    alone = run(*args, reverse=True, noise=np.random.default_rng(1))
    rows.append({"graph": name, "halo": info["halo"], "input": "x".join(str(d or "any") for d in info["dims"][2:]),
                 "told": info["condition"] is not None, "largest core change": float(np.abs(packed - alone).max()),
                 "ms a patch": round(1000 * (time.time() - start) / (2 * len(sample)), 2)})
checked = pd.DataFrame(rows)
print(checked.to_string(index=False))
assert (checked["largest core change"] < 2e-3).all(), "a core changed with what is beside it: packing is not exact for that graph"

                                             graph  halo   input  told  largest core change  ms a patch
               upscale_static_nearest_tile128.onnx     0 128x128 False                  0.0        0.34
          upscale_static_bilinear_tile320x180.onnx     1 182x322 False                  0.0        0.23
           upscale_static_bicubic_tile320x180.onnx     2 184x324 False                  0.0        5.05
                           upscale_nn_tile128.onnx     4 136x136 False                  0.0        1.22
                          upscale_web_tile128.onnx     4 136x136 False                  0.0        1.03
                     upscale_web_int8_tile128.onnx     4 136x136 False                  0.0        1.14
     upscale_strategy_32-16_leaky_tile320x180.onnx     4 188x328 False                  0.0        0.52
upscale_strategy_luma_24_2_blocks_tile320x180.onnx     4 188x328 False                  0.0        0.88
      upscale_strategy_luma_32-16_tile320x180.onnx     4 188x328

## 3. Scores

Each graph on every patch of both sets, a chunk at a time, scored on the GPU with `metrics.per_image`; and
`models.bicubic`, the reference. Per-patch scores are cached under the graph's hash, so a re-exported graph is scored
again and the rest are read back.

In [5]:
@torch.no_grad()
def per_patch(prediction, hr):
    """SSIM-Y and luma PSNR of each core against its target."""
    target = torch.from_numpy(hr).to(DEVICE).permute(0, 3, 1, 2).float() / 255
    found = metrics.per_image(torch.as_tensor(prediction).to(DEVICE), target)
    return found["ssim"].cpu().numpy(), found["y"].cpu().numpy()


@torch.no_grad()
def bicubic(lr, d):
    """models.bicubic's core for LR patches."""
    x = torch.from_numpy(lr).to(DEVICE).permute(0, 3, 1, 2).float() / 255
    return patches.trim(models.bicubic(x, d["scale"]), d["halo"] * d["scale"])


def score(key, name):
    """{"ssim", "y"} of one graph (or "bicubic") on every patch of a set, from the cache if it is there."""
    d = data[key]
    digest = "bicubic" if name == "bicubic" else hashlib.md5((DIST / name).read_bytes()).hexdigest()[:12]
    cached = CACHE / f"{key}_{name.replace('/', '__')}_{digest}.npz"
    if cached.exists():
        with np.load(cached) as found:
            return {"ssim": found["ssim"], "y": found["y"]}
    if name != "bicubic":
        info, graph = graph_info(DIST / name), open_graph(DIST / name)
    ssim, y = [], []
    for first in range(0, len(d["lr"]), CHUNK):
        members = np.arange(first, min(first + CHUNK, len(d["lr"])))
        if name == "bicubic":
            prediction = bicubic(d["lr"][members], d)
        else:
            prediction = run(graph, info, d["lr"][members], conditions_of(d, members), d["halo"], d["scale"])
        s, v = per_patch(prediction, d["hr"][members])
        ssim.append(s)
        y.append(v)
    found = {"ssim": np.concatenate(ssim), "y": np.concatenate(y)}
    np.savez(cached, **found)
    return found


results = {}
start = time.time()
for key in SETS:
    for name in ["bicubic"] + [name for _, name, _ in GRAPHS]:
        began = time.time()
        results[key, name] = score(key, name)
        print(f"  notebook {key}'s set  {name:<52} SSIM-Y {results[key, name]['ssim'].mean():.4f}  {time.time() - began:5.0f} s")
print(f"{time.time() - start:.0f} s")

  notebook 28's set  bicubic                                              SSIM-Y 0.8524      1 s
  notebook 28's set  upscale_static_nearest_tile128.onnx                  SSIM-Y 0.8365      3 s
  notebook 28's set  upscale_static_bilinear_tile320x180.onnx             SSIM-Y 0.8437      4 s
  notebook 28's set  upscale_static_bicubic_tile320x180.onnx              SSIM-Y 0.8524    118 s
  notebook 28's set  upscale_nn_tile128.onnx                              SSIM-Y 0.8650     31 s
  notebook 28's set  upscale_web_tile128.onnx                             SSIM-Y 0.8710     26 s
  notebook 28's set  upscale_web_int8_tile128.onnx                        SSIM-Y 0.8696     25 s
  notebook 28's set  upscale_strategy_32-16_leaky_tile320x180.onnx        SSIM-Y 0.8709     13 s
  notebook 28's set  upscale_strategy_luma_24_2_blocks_tile320x180.onnx   SSIM-Y 0.8763     17 s
  notebook 28's set  upscale_strategy_luma_32-16_tile320x180.onnx         SSIM-Y 0.8706     14 s
  notebook 28's set  upscale_r

### The check: notebook 21's set, against the numbers recorded on it

In [6]:
rows = []
for name in ["bicubic"] + [name for _, name, _ in GRAPHS]:
    found = results["21", name]
    row = {"graph": name, "SSIM-Y": round(float(found["ssim"].mean()), 4), "luma dB": round(float(found["y"].mean()), 2)}
    if name in RECORDED_21:
        row["recorded SSIM-Y"], row["recorded dB"] = RECORDED_21[name]
    rows.append(row)
table = pd.DataFrame(rows).fillna("")
print(table[table["recorded SSIM-Y"] != ""].to_string(index=False))

                   graph  SSIM-Y  luma dB recorded SSIM-Y recorded dB
                 bicubic  0.9045    33.55          0.9045       33.55
     client/upscale.onnx  0.9289    35.07          0.9289       35.07
client/upscale_fast.onnx  0.9213    34.47          0.9212       34.47


## 4. The scoreboard

Every model on notebook 28's validation split. **SSIM-Y** and **luma dB** are means over the 24,216 patches; **over
bicubic** is the mean gain against `models.bicubic` on the same patches, with its 95% range over sources; then the
SSIM-Y gain by kind and by encoder.

In [7]:
def gain_range(key, name, field="ssim"):
    """95% range of the mean gain over bicubic, from a bootstrap over the set's sources."""
    d = data[key]
    gain = results[key, name][field] - results[key, "bicubic"][field]
    count = np.bincount(d["source"])
    total = np.bincount(d["source"], weights=gain)
    draws = np.random.default_rng(0).multinomial(len(count), np.full(len(count), 1 / len(count)), size=BOOTSTRAP)
    means = (draws @ total) / (draws @ count)
    return np.quantile(means, [0.025, 0.975])


def board(key):
    d = data[key]
    base = results[key, "bicubic"]
    rows = [{"nb": "", "model": "bicubic (models.bicubic)", "SSIM-Y": base["ssim"].mean(), "over bicubic": 0.0, "95% range": "",
             "luma dB": base["y"].mean(), "dB over": 0.0}]
    for notebook, name, label in GRAPHS:
        found = results[key, name]
        gain = found["ssim"] - base["ssim"]
        low, high = gain_range(key, name)
        row = {"nb": notebook, "model": label, "SSIM-Y": found["ssim"].mean(), "over bicubic": gain.mean(),
               "95% range": f"{low:+.4f} to {high:+.4f}", "luma dB": found["y"].mean(), "dB over": (found["y"] - base["y"]).mean()}
        for kind in dict.fromkeys(d["kind"]):
            row[kind] = gain[d["kind"] == kind].mean()
        for encoder in dict.fromkeys(d["encoder_name"]):
            row[encoder] = gain[d["encoder_name"] == encoder].mean()
        rows.append(row)
    table = pd.DataFrame(rows).sort_values("SSIM-Y", ascending=False)
    return table


scoreboard = board("28")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}" if abs(v) < 5 else f"{v:.2f}"))

nb                       model  SSIM-Y  over bicubic          95% range  luma dB  dB over   video  screen   photo    x264   nvenc
23    told 24x2 (the client's)  0.8878        0.0354 +0.0337 to +0.0371    30.14   1.5353  0.0249  0.0474  0.0257  0.0381  0.0327
 8  screens 24x2, told 0.2 bpp  0.8826        0.0301 +0.0280 to +0.0324    30.05   1.4453  0.0146  0.0483  0.0149  0.0299  0.0304
 8 screens 24x2, told 0.46 bpp  0.8820        0.0296 +0.0274 to +0.0319    30.07   1.4696  0.0132  0.0481  0.0153  0.0275  0.0316
10               LUT SDY x2 17  0.8797        0.0273 +0.0248 to +0.0299    29.81   1.2125  0.0078  0.0493  0.0105  0.0229  0.0316
 8    screens 24x2, told 1 bpp  0.8789        0.0265 +0.0242 to +0.0288    30.02   1.4235  0.0095  0.0456  0.0121  0.0221  0.0307
16 luma 32-16 leaky, distilled  0.8770        0.0246 +0.0224 to +0.0269    29.66   1.0563  0.0077  0.0433  0.0112  0.0202  0.0289
 4 strategy luma 24 + 2 blocks  0.8763        0.0239 +0.0224 to +0.0254    29.75   1.1539 

### By slice QP

The SSIM-Y gain over bicubic in each slice QP bin, P and I frames together: where each model helps and where it
hurts. The bins' patch counts are in the first row.

In [8]:
d = data["28"]
bins = np.digitize(d["qp"], QP_EDGES[1:-1])
labels = [f"{a}-{b}" for a, b in zip(QP_EDGES[:-1], QP_EDGES[1:])]
rows = [{"model": "patches", **{label: float((bins == i).sum()) for i, label in enumerate(labels)}}]
base = results["28", "bicubic"]["ssim"]
order = {label: rank for rank, label in enumerate(scoreboard["model"])}
for notebook, name, label in sorted(GRAPHS, key=lambda g: order[g[2]]):
    gain = results["28", name]["ssim"] - base
    rows.append({"model": f"{label} ({notebook})", **{l: gain[bins == i].mean() for i, l in enumerate(labels)}})
with pd.option_context("display.width", 250):
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:+.4f}" if abs(v) < 1 else f"{v:.0f}"))

                           model    0-18   18-26   26-30   30-34   34-38   38-42   42-46   46-52
                         patches    3616    3908    3704    3666    3567    2825    1946     984
   told 24x2 (the client's) (23) +0.0315 +0.0336 +0.0344 +0.0361 +0.0387 +0.0401 +0.0357 +0.0314
  screens 24x2, told 0.2 bpp (8) +0.0304 +0.0341 +0.0345 +0.0334 +0.0307 +0.0259 +0.0197 +0.0159
 screens 24x2, told 0.46 bpp (8) +0.0346 +0.0371 +0.0353 +0.0317 +0.0270 +0.0215 +0.0157 +0.0123
              LUT SDY x2 17 (10) +0.0365 +0.0370 +0.0331 +0.0282 +0.0230 +0.0169 +0.0111 +0.0075
    screens 24x2, told 1 bpp (8) +0.0369 +0.0370 +0.0324 +0.0267 +0.0209 +0.0153 +0.0103 +0.0074
luma 32-16 leaky, distilled (16) +0.0325 +0.0335 +0.0301 +0.0255 +0.0205 +0.0153 +0.0103 +0.0065
 strategy luma 24 + 2 blocks (4) +0.0291 +0.0305 +0.0280 +0.0246 +0.0208 +0.0175 +0.0132 +0.0105
                   luma 24x2 (7) +0.0291 +0.0305 +0.0280 +0.0246 +0.0208 +0.0175 +0.0132 +0.0105
                DirNet full (1

## 5. What it says

**The check holds.** On notebook 21's set this code gives bicubic 0.9045 SSIM-Y / 33.55 dB, the told 24×2 0.9289 /
35.07 and q24 0.9213 / 34.47, against the recorded 0.9045, 0.9289 and 0.9212. Packing patches into a tile changed no
core of any graph (section 2), and every graph in `dist/upscale/` is scored or is a twin of one that is.

**On one set, the client's told 24×2 (notebook 23) is the best model: +0.0354 SSIM-Y over bicubic** (+0.0337 to
+0.0371), clear of notebook 8's graphs at +0.0265 to +0.0301, whose ranges end below its. It is also the only model
whose gain holds as the slice QP rises, +0.031 to +0.040 in every bin, with q24, also told the QP. Every model that
is not told it (notebook 8's were told a bitrate once, when exported) keeps at QP 46–52 about half its gain at 0–18 or
less: they were trained on notebook 0's x264 streams at 0.2–1 bpp, which seldom reach those QPs.

**By kind.** On screens the leaders are within 0.004 of each other: notebook 10's LUT SDY x2 (+0.0493), notebook 8's
graphs (+0.0456 to +0.0483) and the told 24×2 (+0.0474); no range by kind was computed to separate them. On video and photos the told 24×2 leads: +0.0249 on video (next, q24 at
+0.0176) and +0.0257 on photos (next, notebook 7's luma 24×2 at +0.0176).

**q24 (notebook 24, the client's fallback for slower GPUs) is mid-table at +0.0186**, level with notebook 3's web
32-16. It is weakest where the picture is cleanest, +0.0139 at QP 0–18 and +0.0212 on screens, where the larger models
gain +0.03 to +0.05. Like the told 24×2, it holds its gain at high QP.

**Four graphs do little or nothing.** Notebook 6's reparam 32-16 scores bicubic to four places in every bin: its
published graph is bicubic. Notebook 9's DCT fixed and notebook 12's DirNet fixed gain under +0.002 overall and fall
below bicubic from QP 30–34 up. Notebook 1's bicubic `Resize` graph scores what `models.bicubic` does. Notebook 7's
luma 24×2 and notebook 4's luma 24 + 2 blocks score the same to every digit: notebook 7 exports notebook 4's run.

**From here on**, a model is scored on this set by adding its graph to `GRAPHS` and running the notebook again;
graphs already scored are read back from `scoreboard/`.